# 08 - Association Rules by Cluster

This notebook mines simple pairwise association rules inside each final K-Means cluster. Basket data is used only after the final customer clusters are assigned.

## Scope

- Read the final `outputs/customer_clusters.csv` file only; do not modify it.
- Use `data/raw/customer_basket.csv` only for post-clustering purchasing behaviour.
- Mine pairwise association rules by cluster.
- Do not create promotion recommendations in this phase.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("../src")

from promotions import (
    add_basket_level_features,
    create_cluster_association_rules,
    join_baskets_to_clusters,
    validate_customer_clusters,
)

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_colwidth", 90)

## Load Data

Both inputs are loaded with direct `pd.read_csv` calls. The final cluster assignment is read only.

In [ ]:
customer_basket = pd.read_csv("../data/raw/customer_basket.csv")
customer_clusters = pd.read_csv("../outputs/customer_clusters.csv")

print("customer_basket:", customer_basket.shape)
print("customer_clusters:", customer_clusters.shape)
customer_clusters.head()

## Validate Final Cluster File

The association-rule phase depends on the locked final segmentation output. These checks make sure the notebook is using the expected final file.

In [ ]:
validate_customer_clusters(customer_clusters)

assert customer_clusters.columns.tolist() == ["customer_id", "cluster"]
assert len(customer_clusters) == 33038
assert customer_clusters["customer_id"].duplicated().sum() == 0
assert customer_clusters["cluster"].isna().sum() == 0

print("Final cluster file validation passed.")

## Prepare Basket Records

The basket table is parsed into product lists, then joined to the final customer clusters. Customers without basket records stay in `customer_clusters`; they simply cannot contribute to basket association rules.

In [ ]:
basket_features = add_basket_level_features(customer_basket)
basket_clusters = join_baskets_to_clusters(basket_features, customer_clusters)

assert basket_clusters["cluster"].notna().all()

customers_with_baskets = basket_clusters["customer_id"].nunique()
customers_without_baskets = len(customer_clusters) - customers_with_baskets

print("Basket rows with assigned cluster:", len(basket_clusters))
print("Customers with basket records:", customers_with_baskets)
print("Clustered customers without basket records:", customers_without_baskets)

basket_clusters[["invoice_id", "customer_id", "cluster", "goods_list", "basket_size"]].head()

## Mine Cluster-Level Rules

The thresholds are intentionally moderate so each cluster keeps interpretable rules without creating a large experimental output. Rules are ranked by lift, then confidence, then support.

In [ ]:
MIN_SUPPORT = 0.01
MIN_CONFIDENCE = 0.15
MIN_LIFT = 1.20
TOP_N_PER_CLUSTER = 20

cluster_rules = create_cluster_association_rules(
    basket_clusters,
    min_support=MIN_SUPPORT,
    min_confidence=MIN_CONFIDENCE,
    min_lift=MIN_LIFT,
    max_len=2,
    top_n=TOP_N_PER_CLUSTER,
)

cluster_rules.to_csv("../outputs/cluster_association_rules.csv", index=False)

print("Saved rows:", len(cluster_rules))
cluster_rules.head()

## Rule Strength by Cluster

Each cluster is capped at the top 20 rules, so the number of displayed rules is not a measure of total association richness. Lift and confidence are more useful for comparing rule strength.

In [ ]:
rule_summary = (
    cluster_rules.groupby("cluster")
    .agg(
        number_of_saved_rules=("rule_rank", "count"),
        average_lift=("lift", "mean"),
        max_lift=("lift", "max"),
        average_confidence=("confidence", "mean"),
        max_confidence=("confidence", "max"),
    )
    .reset_index()
)

rule_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=rule_summary, x="cluster", y="max_lift", ax=ax, color="#5DA5DA")
ax.set_title("Strongest Association Rule Lift by Cluster")
ax.set_xlabel("Cluster")
ax.set_ylabel("Max lift")
plt.tight_layout()

## Top Rules

The table below shows the strongest rules per cluster. A high lift means the consequent appears more often with the antecedent than expected from its baseline frequency.

In [ ]:
top_rules = cluster_rules.sort_values(["cluster", "rule_rank"]).groupby("cluster").head(5).copy()
top_rules["rule"] = top_rules["antecedents"] + " -> " + top_rules["consequents"]

top_rules[
    ["cluster", "rule_rank", "rule", "support", "confidence", "lift"]
].round({"support": 4, "confidence": 4, "lift": 4})

## Interpretation

- Cluster 0 has strong electronics and drink co-occurrence, especially `bluetooth headphones`, `airpods`, and `energy drink`.
- Cluster 1 combines a similar electronics and energy-drink signal with fresh-product pairings such as `salad` and `avocado`.
- Cluster 2 has weaker but still useful rules around `energy drink`, `airpods`, and practical grocery/home pairs like `cooking oil` and `napkins`.
- Cluster 3 has the strongest lifts, dominated by electronics accessory combinations such as `airpods`, `bluetooth headphones`, and `laptop`.
- Cluster 4 has lower lift overall and is more grocery-oriented, with everyday product pairings rather than a sharp electronics bundle.

## Caveats

- These rules describe observed baskets only; customers without basket records remain part of the final cluster output but do not contribute to rule mining.
- Association rules show co-occurrence, not causality.
- This notebook does not create recommendations yet. The next phase should translate these findings into simple cluster-level promotion candidates.

## Validation

In [ ]:
saved_rules = pd.read_csv("../outputs/cluster_association_rules.csv")

expected_columns = [
    "cluster",
    "antecedents",
    "consequents",
    "antecedent_support",
    "consequent_support",
    "support",
    "confidence",
    "lift",
    "leverage",
    "conviction",
    "rule_rank",
]

assert saved_rules.columns.tolist() == expected_columns
assert saved_rules.shape == cluster_rules.shape
assert saved_rules["cluster"].nunique() == customer_clusters["cluster"].nunique()
assert Path("../outputs/cluster_association_rules.csv").exists()

print("Association-rule output validation passed.")